In [ ]:
dataset_root = Path(r"C:\Users\John joseph peter\Desktop\NUS\ApneaSense\SLP2022\SLP\danaLab")

In [1]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

2.10.0+cu128
12.8
True
NVIDIA GeForce RTX 4080 Laptop GPU


In [2]:
import torchvision
import os
import pandas as pd
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [13]:
import scipy.io as sio
import numpy as np

mat = sio.loadmat(str(dataset_root / "00001" / "joints_gt_RGB.mat"))
j = mat["joints_gt"]

print("overall shape:", j.shape)

# overall ranges across all frames
print("overall x min/max:", j[0].min(), j[0].max())
print("overall y min/max:", j[1].min(), j[1].max())
print("overall channel2 unique:", np.unique(j[2]))

# how many nonzero values in channel 2 per frame
for k in range(5):
    nonzero = np.count_nonzero(j[2, :, k])
    print(f"frame {k}: channel2 nonzero joints = {nonzero} / 14")

overall shape: (3, 14, 45)
overall x min/max: 116.22327044025153 473.65723270440253
overall y min/max: 114.81446540880506 829.682389937107
overall channel2 unique: [0. 1.]
frame 0: channel2 nonzero joints = 0 / 14
frame 1: channel2 nonzero joints = 0 / 14
frame 2: channel2 nonzero joints = 0 / 14
frame 3: channel2 nonzero joints = 0 / 14
frame 4: channel2 nonzero joints = 0 / 14


In [14]:
print("shape:", j.shape)

for k in range(j.shape[2]):
    ch2 = j[2, :, k]
    print(
        f"frame {k+1:02d} | nonzero={np.count_nonzero(ch2):2d}/14 | values={ch2.astype(int)}"
    )

shape: (3, 14, 45)
frame 01 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 02 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 03 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 04 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 05 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 06 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 07 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 08 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 09 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 10 | nonzero= 1/14 | values=[0 0 0 0 0 1 0 0 0 0 0 0 0 0]
frame 11 | nonzero= 1/14 | values=[0 0 0 0 0 1 0 0 0 0 0 0 0 0]
frame 12 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 13 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 14 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 15 | nonzero= 0/14 | values=[0 0 0 0 0 0 0 0 0 0 0 0 0 0]
frame 16 | nonzero= 5

Test 1, first try. Only XY no occluded input. (Noted Duplication)

In [20]:
# train_joint_mlp_v2.py
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import scipy.io as sio

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


@dataclass
class Config:
    dataset_root: str = r"C:/Users/John joseph peter/Desktop/NUS/ApneaSense/SLP2022/SLP/danaLab"
    csv_path: str = r"C:/Users/John joseph peter/Desktop/NUS/ApneaSense/SLP2022/SLP/danaLab/posture_labels_all_modalities.csv"
    joint_source: str = "RGB"   # "RGB" or "IR"

    batch_size: int = 64
    num_epochs: int = 20
    lr: float = 1e-3
    weight_decay: float = 1e-4
    dropout: float = 0.3
    hidden_dim1: int = 128
    hidden_dim2: int = 128
    num_workers: int = 0
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    save_dir: str = "./checkpoints_joint_mlp_v2"


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_save_dir(path: str):
    os.makedirs(path, exist_ok=True)


def load_joint_mat(mat_path: str) -> np.ndarray:
    data = sio.loadmat(mat_path)
    return data["joints_gt"]   # (3, 14, 45)


def build_joint_metadata(csv_path: str, joint_source: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # force subject_id back to 5-digit folder format
    df["subject_id"] = df["subject_id"].astype(int).astype(str).str.zfill(5)

    # use one modality only as reference rows
    df = df[df["modality"] == "RGB"].copy()

    joint_filename = {
        "RGB": "joints_gt_RGB.mat",
        "IR": "joints_gt_IR.mat"
    }[joint_source]

    df["joint_file"] = df["subject_id"].apply(lambda s: f"{s}/{joint_filename}")
    df["frame_idx_0based"] = df["image_index"] - 1

    return df[[
        "subject_id",
        "condition",
        "image_index",
        "frame_idx_0based",
        "joint_file",
        "label",
        "label_id"
    ]].reset_index(drop=True)


def subject_wise_split(subject_ids: List[str], cfg: Config):
    subject_ids = sorted(subject_ids)
    random.shuffle(subject_ids)

    n = len(subject_ids)
    n_train = int(n * cfg.train_ratio)
    n_val = int(n * cfg.val_ratio)

    train_subjects = subject_ids[:n_train]
    val_subjects = subject_ids[n_train:n_train + n_val]
    test_subjects = subject_ids[n_train + n_val:]

    return train_subjects, val_subjects, test_subjects


def compute_train_xy_scale(train_df: pd.DataFrame, dataset_root: str) -> Tuple[float, float]:
    """
    Compute global max x and y from the TRAIN set only.
    """
    joint_cache: Dict[str, np.ndarray] = {}
    max_x, max_y = 1.0, 1.0

    for _, row in train_df.iterrows():
        joint_rel = row["joint_file"]
        if joint_rel not in joint_cache:
            joint_cache[joint_rel] = load_joint_mat(os.path.join(dataset_root, joint_rel))

        joints_all = joint_cache[joint_rel]
        f = int(row["frame_idx_0based"])

        frame = joints_all[:, :, f]   # (3, 14)
        x = frame[0]
        y = frame[1]

        max_x = max(max_x, float(np.max(x)))
        max_y = max(max_y, float(np.max(y)))

    return max_x, max_y


def preprocess_joint_frame_xy_only(frame_joints: np.ndarray, max_x: float, max_y: float) -> np.ndarray:
    """
    frame_joints: (3, 14)
    We ignore channel 2 for now because it looks unreliable in some frames.
    Use only x,y -> output shape (28,)
    """
    x = frame_joints[0].astype(np.float32) / max_x
    y = frame_joints[1].astype(np.float32) / max_y

    out = np.stack([x, y], axis=1)   # (14, 2)
    return out.reshape(-1)           # (28,)


class JointFrameDataset(Dataset):
    def __init__(self, df: pd.DataFrame, dataset_root: str, max_x: float, max_y: float):
        self.df = df.reset_index(drop=True)
        self.dataset_root = dataset_root
        self.max_x = max_x
        self.max_y = max_y
        self.mat_cache: Dict[str, np.ndarray] = {}

    def __len__(self):
        return len(self.df)

    def _get_subject_joints(self, joint_file_rel: str) -> np.ndarray:
        if joint_file_rel not in self.mat_cache:
            abs_path = os.path.join(self.dataset_root, joint_file_rel)
            self.mat_cache[joint_file_rel] = load_joint_mat(abs_path)
        return self.mat_cache[joint_file_rel]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        joints_all = self._get_subject_joints(row["joint_file"])   # (3, 14, 45)
        frame = joints_all[:, :, int(row["frame_idx_0based"])]     # (3, 14)

        x = preprocess_joint_frame_xy_only(frame, self.max_x, self.max_y)
        y = int(row["label_id"])

        return {
            "x": torch.tensor(x, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.long),
            "condition": row["condition"],
            "subject_id": row["subject_id"]
        }


class JointMLP(nn.Module):
    def __init__(self, input_dim=28, hidden_dim1=128, hidden_dim2=128, dropout=0.3, num_classes=3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Linear(hidden_dim2, num_classes)

    def forward(self, x, return_features=False):
        feats = self.encoder(x)
        logits = self.classifier(feats)
        if return_features:
            return logits, feats
        return logits


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    y_true, y_pred = [], []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        pred = torch.argmax(logits, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())

    return (
        total_loss / len(loader.dataset),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
    )


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    conditions = []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)

        pred = torch.argmax(logits, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())
        conditions.extend(batch["condition"])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    conditions = np.array(conditions)

    per_condition = {}
    for cond in sorted(np.unique(conditions)):
        mask = conditions == cond
        per_condition[cond] = {
            "acc": accuracy_score(y_true[mask], y_pred[mask]),
            "f1_macro": f1_score(y_true[mask], y_pred[mask], average="macro")
        }

    return (
        total_loss / len(loader.dataset),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
        confusion_matrix(y_true, y_pred),
        per_condition
    )


def print_metrics(name, loss, acc, f1, per_condition=None):
    print(f"\n[{name}]")
    print(f"Loss     : {loss:.4f}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Macro F1 : {f1:.4f}")

    if per_condition is not None:
        for cond, m in per_condition.items():
            print(f"  {cond:8s} | acc={m['acc']:.4f} | f1={m['f1_macro']:.4f}")


def main():
    cfg = Config()
    set_seed(cfg.seed)
    make_save_dir(cfg.save_dir)

    df = build_joint_metadata(cfg.csv_path, cfg.joint_source)
    subjects = sorted(df["subject_id"].unique().tolist())

    train_subjects, val_subjects, test_subjects = subject_wise_split(subjects, cfg)

    train_df = df[df["subject_id"].isin(train_subjects)].reset_index(drop=True)
    val_df = df[df["subject_id"].isin(val_subjects)].reset_index(drop=True)
    test_df = df[df["subject_id"].isin(test_subjects)].reset_index(drop=True)

    print("train samples:", len(train_df))
    print("val samples  :", len(val_df))
    print("test samples :", len(test_df))

    max_x, max_y = compute_train_xy_scale(train_df, cfg.dataset_root)
    print("train max_x:", max_x)
    print("train max_y:", max_y)

    train_ds = JointFrameDataset(train_df, cfg.dataset_root, max_x, max_y)
    val_ds = JointFrameDataset(val_df, cfg.dataset_root, max_x, max_y)
    test_ds = JointFrameDataset(test_df, cfg.dataset_root, max_x, max_y)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

    model = JointMLP(
        input_dim=28,
        hidden_dim1=cfg.hidden_dim1,
        hidden_dim2=cfg.hidden_dim2,
        dropout=cfg.dropout,
        num_classes=3
    ).to(cfg.device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val_f1 = -1
    ckpt_path = os.path.join(cfg.save_dir, f"best_joint_mlp_xyonly_{cfg.joint_source}.pth")

    for epoch in range(cfg.num_epochs):
        tr_loss, tr_acc, tr_f1 = train_one_epoch(model, train_loader, optimizer, criterion, cfg.device)
        va_loss, va_acc, va_f1, va_cm, va_pc = evaluate(model, val_loader, criterion, cfg.device)

        print(f"\nEpoch {epoch+1}/{cfg.num_epochs}")
        print_metrics("Train", tr_loss, tr_acc, tr_f1)
        print_metrics("Val", va_loss, va_acc, va_f1, va_pc)

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "max_x": max_x,
                    "max_y": max_y,
                    "config": cfg.__dict__
                },
                ckpt_path
            )
            print("saved:", ckpt_path)

    print("\nLoading best model for test...")
    ckpt = torch.load(ckpt_path, map_location=cfg.device)
    model.load_state_dict(ckpt["model_state_dict"])

    te_loss, te_acc, te_f1, te_cm, te_pc = evaluate(model, test_loader, criterion, cfg.device)
    print_metrics("Test", te_loss, te_acc, te_f1, te_pc)
    print("\nConfusion matrix:")
    print(te_cm)


if __name__ == "__main__":
    main()

train samples: 9585
val samples  : 2025
test samples : 2160
train max_x: 519.7807991120978
train max_y: 893.7252591894439

Epoch 1/20

[Train]
Loss     : 0.5070
Accuracy : 0.8533
Macro F1 : 0.8544

[Val]
Loss     : 0.0755
Accuracy : 0.9867
Macro F1 : 0.9866
  cover1   | acc=0.9867 | f1=0.9866
  cover2   | acc=0.9867 | f1=0.9866
  uncover  | acc=0.9867 | f1=0.9866
saved: ./checkpoints_joint_mlp_v2\best_joint_mlp_xyonly_RGB.pth

Epoch 2/20

[Train]
Loss     : 0.1082
Accuracy : 0.9678
Macro F1 : 0.9678

[Val]
Loss     : 0.0367
Accuracy : 0.9896
Macro F1 : 0.9896
  cover1   | acc=0.9896 | f1=0.9896
  cover2   | acc=0.9896 | f1=0.9896
  uncover  | acc=0.9896 | f1=0.9896
saved: ./checkpoints_joint_mlp_v2\best_joint_mlp_xyonly_RGB.pth

Epoch 3/20

[Train]
Loss     : 0.0712
Accuracy : 0.9787
Macro F1 : 0.9787

[Val]
Loss     : 0.0270
Accuracy : 0.9896
Macro F1 : 0.9896
  cover1   | acc=0.9896 | f1=0.9896
  cover2   | acc=0.9896 | f1=0.9896
  uncover  | acc=0.9896 | f1=0.9896
saved: ./checkpoin

Test 2. XYO. (Deduplication)

In [4]:
# train_joint_mlp_v2.py
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import scipy.io as sio

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


@dataclass
class Config:
    dataset_root: str = r"C:/Users/John joseph peter/Desktop/NUS/ApneaSense/SLP2022/SLP/danaLab"
    csv_path: str = r"C:/Users/John joseph peter/Desktop/NUS/ApneaSense/SLP2022/SLP/danaLab/posture_labels_all_modalities.csv"
    joint_source: str = "RGB"   # "RGB" or "IR"

    batch_size: int = 64
    num_epochs: int = 20
    lr: float = 1e-3
    weight_decay: float = 1e-4
    dropout: float = 0.3
    hidden_dim1: int = 128
    hidden_dim2: int = 128
    num_workers: int = 0
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    save_dir: str = "./checkpoints_joint_mlp_v2"


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_save_dir(path: str):
    os.makedirs(path, exist_ok=True)


def load_joint_mat(mat_path: str) -> np.ndarray:
    data = sio.loadmat(mat_path)
    return data["joints_gt"]   # (3, 14, 45)


def build_joint_metadata(csv_path: str, joint_source: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # force subject_id back to 5-digit folder format
    df["subject_id"] = df["subject_id"].astype(int).astype(str).str.zfill(5)

    # use one modality only as reference rows
    df = df[df["modality"] == "RGB"].copy()

    # keep only one row per subject + frame index
    df = df.drop_duplicates(subset=["subject_id", "image_index"]).copy()

    joint_filename = {
        "RGB": "joints_gt_RGB.mat",
        "IR": "joints_gt_IR.mat"
    }[joint_source]

    df["joint_file"] = df["subject_id"].apply(lambda s: f"{s}/{joint_filename}")
    df["frame_idx_0based"] = df["image_index"] - 1

    return df[[
        "subject_id",
        "image_index",
        "frame_idx_0based",
        "joint_file",
        "label",
        "label_id"
    ]].reset_index(drop=True)


def subject_wise_split(subject_ids: List[str], cfg: Config):
    subject_ids = sorted(subject_ids)
    random.shuffle(subject_ids)

    n = len(subject_ids)
    n_train = int(n * cfg.train_ratio)
    n_val = int(n * cfg.val_ratio)

    train_subjects = subject_ids[:n_train]
    val_subjects = subject_ids[n_train:n_train + n_val]
    test_subjects = subject_ids[n_train + n_val:]

    return train_subjects, val_subjects, test_subjects


def compute_train_xy_scale(train_df: pd.DataFrame, dataset_root: str) -> Tuple[float, float]:
    """
    Compute global max x and y from the TRAIN set only.
    """
    joint_cache: Dict[str, np.ndarray] = {}
    max_x, max_y = 1.0, 1.0

    for _, row in train_df.iterrows():
        joint_rel = row["joint_file"]
        if joint_rel not in joint_cache:
            joint_cache[joint_rel] = load_joint_mat(os.path.join(dataset_root, joint_rel))

        joints_all = joint_cache[joint_rel]
        f = int(row["frame_idx_0based"])

        frame = joints_all[:, :, f]   # (3, 14)
        x = frame[0]
        y = frame[1]

        max_x = max(max_x, float(np.max(x)))
        max_y = max(max_y, float(np.max(y)))

    return max_x, max_y


# def preprocess_joint_frame_xy_only(frame_joints: np.ndarray, max_x: float, max_y: float) -> np.ndarray:
#     """
#     frame_joints: (3, 14)
#     We ignore channel 2 for now because it looks unreliable in some frames.
#     Use only x,y -> output shape (28,)
#     """
#     x = frame_joints[0].astype(np.float32) / max_x
#     y = frame_joints[1].astype(np.float32) / max_y

#     out = np.stack([x, y], axis=1)   # (14, 2)
#     return out.reshape(-1)           # (28,)
def preprocess_joint_frame_xyo(frame_joints: np.ndarray) -> np.ndarray:
    """
    frame_joints: (3, 14)
    channels:
      0 -> x
      1 -> y
      2 -> if_occluded
    output: (14*3,) = 42
    """
    x = frame_joints[0].astype(np.float32) / 576.0
    y = frame_joints[1].astype(np.float32) / 1024.0
    occ = frame_joints[2].astype(np.float32)   # already 0/1

    out = np.stack([x, y, occ], axis=1)   # (14, 3)
    return out.reshape(-1)                # (42,)

class JointFrameDataset(Dataset):
    def __init__(self, df: pd.DataFrame, dataset_root: str, max_x: float, max_y: float):
        self.df = df.reset_index(drop=True)
        self.dataset_root = dataset_root
        self.max_x = max_x
        self.max_y = max_y
        self.mat_cache: Dict[str, np.ndarray] = {}

    def __len__(self):
        return len(self.df)

    def _get_subject_joints(self, joint_file_rel: str) -> np.ndarray:
        if joint_file_rel not in self.mat_cache:
            abs_path = os.path.join(self.dataset_root, joint_file_rel)
            self.mat_cache[joint_file_rel] = load_joint_mat(abs_path)
        return self.mat_cache[joint_file_rel]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        joints_all = self._get_subject_joints(row["joint_file"])   # (3, 14, 45)
        frame = joints_all[:, :, int(row["frame_idx_0based"])]     # (3, 14)

        x = preprocess_joint_frame_xyo(frame)
        y = int(row["label_id"])

        return {
            "x": torch.tensor(x, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.long),
            # "condition": row["condition"],
            "subject_id": row["subject_id"]
        }


class JointMLP(nn.Module):
    def __init__(self, input_dim=42, hidden_dim1=128, hidden_dim2=128, dropout=0.3, num_classes=3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Linear(hidden_dim2, num_classes)

    def forward(self, x, return_features=False):
        feats = self.encoder(x)
        logits = self.classifier(feats)
        if return_features:
            return logits, feats
        return logits


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    y_true, y_pred = [], []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        pred = torch.argmax(logits, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())

    return (
        total_loss / len(loader.dataset),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
    )


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    # conditions = []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)

        pred = torch.argmax(logits, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())
        # conditions.extend(batch["condition"])

    # y_true = np.array(y_true)
    # y_pred = np.array(y_pred)
    # conditions = np.array(conditions)

    # per_condition = {}
    # for cond in sorted(np.unique(conditions)):
    #     mask = conditions == cond
    #     per_condition[cond] = {
    #         "acc": accuracy_score(y_true[mask], y_pred[mask]),
    #         "f1_macro": f1_score(y_true[mask], y_pred[mask], average="macro")
    #     }

    # return (
    #     total_loss / len(loader.dataset),
    #     accuracy_score(y_true, y_pred),
    #     f1_score(y_true, y_pred, average="macro"),
    #     confusion_matrix(y_true, y_pred),
    #     per_condition
    # )
    return (
        total_loss / len(loader.dataset),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
        confusion_matrix(y_true, y_pred),
    )


def print_metrics(name, loss, acc, f1):
    print(f"\n[{name}]")
    print(f"Loss     : {loss:.4f}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Macro F1 : {f1:.4f}")

    # if per_condition is not None:
    #     for cond, m in per_condition.items():
    #         print(f"  {cond:8s} | acc={m['acc']:.4f} | f1={m['f1_macro']:.4f}")


def main():
    cfg = Config()
    set_seed(cfg.seed)
    make_save_dir(cfg.save_dir)

    df = build_joint_metadata(cfg.csv_path, cfg.joint_source)
    subjects = sorted(df["subject_id"].unique().tolist())

    train_subjects, val_subjects, test_subjects = subject_wise_split(subjects, cfg)

    train_df = df[df["subject_id"].isin(train_subjects)].reset_index(drop=True)
    val_df = df[df["subject_id"].isin(val_subjects)].reset_index(drop=True)
    test_df = df[df["subject_id"].isin(test_subjects)].reset_index(drop=True)

    print("train samples:", len(train_df))
    print("val samples  :", len(val_df))
    print("test samples :", len(test_df))

    max_x, max_y = compute_train_xy_scale(train_df, cfg.dataset_root)
    print("train max_x:", max_x)
    print("train max_y:", max_y)

    train_ds = JointFrameDataset(train_df, cfg.dataset_root, max_x, max_y)
    val_ds = JointFrameDataset(val_df, cfg.dataset_root, max_x, max_y)
    test_ds = JointFrameDataset(test_df, cfg.dataset_root, max_x, max_y)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

    model = JointMLP(
        input_dim=42,
        hidden_dim1=cfg.hidden_dim1,
        hidden_dim2=cfg.hidden_dim2,
        dropout=cfg.dropout,
        num_classes=3
    ).to(cfg.device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val_f1 = -1
    ckpt_path = os.path.join(cfg.save_dir, f"best_joint_mlp_xyo_{cfg.joint_source}.pth")

    for epoch in range(cfg.num_epochs):
        tr_loss, tr_acc, tr_f1 = train_one_epoch(model, train_loader, optimizer, criterion, cfg.device)
        va_loss, va_acc, va_f1, va_cm = evaluate(model, val_loader, criterion, cfg.device)

        print(f"\nEpoch {epoch+1}/{cfg.num_epochs}")
        print_metrics("Train", tr_loss, tr_acc, tr_f1)
        print_metrics("Val", va_loss, va_acc, va_f1)

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "max_x": max_x,
                    "max_y": max_y,
                    "config": cfg.__dict__
                },
                ckpt_path
            )
            print("saved:", ckpt_path)

    print("\nLoading best model for test...")
    ckpt = torch.load(ckpt_path, map_location=cfg.device)
    model.load_state_dict(ckpt["model_state_dict"])

    te_loss, te_acc, te_f1, te_cm= evaluate(model, test_loader, criterion, cfg.device)
    print_metrics("Test", te_loss, te_acc, te_f1)
    print("\nConfusion matrix:")
    print(te_cm)


if __name__ == "__main__":
    main()

train samples: 3195
val samples  : 675
test samples : 720
train max_x: 519.7807991120978
train max_y: 893.7252591894439

Epoch 1/20

[Train]
Loss     : 0.6683
Accuracy : 0.7975
Macro F1 : 0.7967

[Val]
Loss     : 0.1319
Accuracy : 0.9778
Macro F1 : 0.9779
saved: ./checkpoints_joint_mlp_v2\best_joint_mlp_xyo_RGB.pth

Epoch 2/20

[Train]
Loss     : 0.1503
Accuracy : 0.9571
Macro F1 : 0.9573

[Val]
Loss     : 0.0589
Accuracy : 0.9837
Macro F1 : 0.9837
saved: ./checkpoints_joint_mlp_v2\best_joint_mlp_xyo_RGB.pth

Epoch 3/20

[Train]
Loss     : 0.1116
Accuracy : 0.9703
Macro F1 : 0.9703

[Val]
Loss     : 0.0446
Accuracy : 0.9911
Macro F1 : 0.9911
saved: ./checkpoints_joint_mlp_v2\best_joint_mlp_xyo_RGB.pth

Epoch 4/20

[Train]
Loss     : 0.1001
Accuracy : 0.9703
Macro F1 : 0.9703

[Val]
Loss     : 0.0432
Accuracy : 0.9896
Macro F1 : 0.9896

Epoch 5/20

[Train]
Loss     : 0.0914
Accuracy : 0.9718
Macro F1 : 0.9719

[Val]
Loss     : 0.0357
Accuracy : 0.9881
Macro F1 : 0.9882

Epoch 6/20

[Tra

In [5]:
# train_joint_mlp_v2.py
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import scipy.io as sio

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


@dataclass
class Config:
    dataset_root: str = r"C:/Users/John joseph peter/Desktop/NUS/ApneaSense/SLP2022/SLP/danaLab"
    csv_path: str = r"C:/Users/John joseph peter/Desktop/NUS/ApneaSense/SLP2022/SLP/danaLab/posture_labels_all_modalities.csv"
    joint_source: str = "RGB"   # "RGB" or "IR"

    batch_size: int = 64
    num_epochs: int = 20
    lr: float = 1e-3
    weight_decay: float = 1e-4
    dropout: float = 0.3
    hidden_dim1: int = 128
    hidden_dim2: int = 128
    num_workers: int = 0
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    save_dir: str = "./checkpoints_joint_mlp_v2"


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_save_dir(path: str):
    os.makedirs(path, exist_ok=True)


def load_joint_mat(mat_path: str) -> np.ndarray:
    data = sio.loadmat(mat_path)
    return data["joints_gt"]   # (3, 14, 45)


def build_joint_metadata(csv_path: str, joint_source: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # force subject_id back to 5-digit folder format
    df["subject_id"] = df["subject_id"].astype(int).astype(str).str.zfill(5)

    # use one modality only as reference rows
    df = df[df["modality"] == "RGB"].copy()

    # DEDUPLICATE: keep only one row per subject + frame index
    df = df.drop_duplicates(subset=["subject_id", "image_index"]).copy()

    joint_filename = {
        "RGB": "joints_gt_RGB.mat",
        "IR": "joints_gt_IR.mat"
    }[joint_source]

    df["joint_file"] = df["subject_id"].apply(lambda s: f"{s}/{joint_filename}")
    df["frame_idx_0based"] = df["image_index"] - 1

    return df[[
        "subject_id",
        "image_index",
        "frame_idx_0based",
        "joint_file",
        "label",
        "label_id"
    ]].reset_index(drop=True)


def subject_wise_split(subject_ids: List[str], cfg: Config):
    subject_ids = sorted(subject_ids)
    random.shuffle(subject_ids)

    n = len(subject_ids)
    n_train = int(n * cfg.train_ratio)
    n_val = int(n * cfg.val_ratio)

    train_subjects = subject_ids[:n_train]
    val_subjects = subject_ids[n_train:n_train + n_val]
    test_subjects = subject_ids[n_train + n_val:]

    return train_subjects, val_subjects, test_subjects


def compute_train_xy_scale(train_df: pd.DataFrame, dataset_root: str) -> Tuple[float, float]:
    """
    Compute global max x and y from the TRAIN set only.
    """
    joint_cache: Dict[str, np.ndarray] = {}
    max_x, max_y = 1.0, 1.0

    for _, row in train_df.iterrows():
        joint_rel = row["joint_file"]
        if joint_rel not in joint_cache:
            joint_cache[joint_rel] = load_joint_mat(os.path.join(dataset_root, joint_rel))

        joints_all = joint_cache[joint_rel]
        f = int(row["frame_idx_0based"])

        frame = joints_all[:, :, f]   # (3, 14)
        x = frame[0]
        y = frame[1]

        max_x = max(max_x, float(np.max(x)))
        max_y = max(max_y, float(np.max(y)))

    return max_x, max_y


def preprocess_joint_frame_xy_only(frame_joints: np.ndarray, max_x: float, max_y: float) -> np.ndarray:
    """
    frame_joints: (3, 14)
    We ignore channel 2 for now because it looks unreliable in some frames.
    Use only x,y -> output shape (28,)
    """
    x = frame_joints[0].astype(np.float32) / max_x
    y = frame_joints[1].astype(np.float32) / max_y

    out = np.stack([x, y], axis=1)   # (14, 2)
    return out.reshape(-1)           # (28,)


class JointFrameDataset(Dataset):
    def __init__(self, df: pd.DataFrame, dataset_root: str, max_x: float, max_y: float):
        self.df = df.reset_index(drop=True)
        self.dataset_root = dataset_root
        self.max_x = max_x
        self.max_y = max_y
        self.mat_cache: Dict[str, np.ndarray] = {}

    def __len__(self):
        return len(self.df)

    def _get_subject_joints(self, joint_file_rel: str) -> np.ndarray:
        if joint_file_rel not in self.mat_cache:
            abs_path = os.path.join(self.dataset_root, joint_file_rel)
            self.mat_cache[joint_file_rel] = load_joint_mat(abs_path)
        return self.mat_cache[joint_file_rel]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        joints_all = self._get_subject_joints(row["joint_file"])   # (3, 14, 45)
        frame = joints_all[:, :, int(row["frame_idx_0based"])]     # (3, 14)

        x = preprocess_joint_frame_xy_only(frame, self.max_x, self.max_y)
        y = int(row["label_id"])

        return {
            "x": torch.tensor(x, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.long),
            "subject_id": row["subject_id"]
        }


class JointMLP(nn.Module):
    def __init__(self, input_dim=28, hidden_dim1=128, hidden_dim2=128, dropout=0.3, num_classes=3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Linear(hidden_dim2, num_classes)

    def forward(self, x, return_features=False):
        feats = self.encoder(x)
        logits = self.classifier(feats)
        if return_features:
            return logits, feats
        return logits


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    y_true, y_pred = [], []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        pred = torch.argmax(logits, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())

    return (
        total_loss / len(loader.dataset),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
    )


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []

    for batch in loader:
        x = batch["x"].to(device)
        y = batch["y"].to(device)

        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)

        pred = torch.argmax(logits, dim=1)
        y_true.extend(y.cpu().numpy())
        y_pred.extend(pred.cpu().numpy())

    return (
        total_loss / len(loader.dataset),
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average="macro"),
        confusion_matrix(y_true, y_pred),
    )


def print_metrics(name, loss, acc, f1):
    print(f"\n[{name}]")
    print(f"Loss     : {loss:.4f}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Macro F1 : {f1:.4f}")


def main():
    cfg = Config()
    set_seed(cfg.seed)
    make_save_dir(cfg.save_dir)

    df = build_joint_metadata(cfg.csv_path, cfg.joint_source)
    subjects = sorted(df["subject_id"].unique().tolist())

    train_subjects, val_subjects, test_subjects = subject_wise_split(subjects, cfg)

    train_df = df[df["subject_id"].isin(train_subjects)].reset_index(drop=True)
    val_df = df[df["subject_id"].isin(val_subjects)].reset_index(drop=True)
    test_df = df[df["subject_id"].isin(test_subjects)].reset_index(drop=True)

    print("train samples:", len(train_df))
    print("val samples  :", len(val_df))
    print("test samples :", len(test_df))

    max_x, max_y = compute_train_xy_scale(train_df, cfg.dataset_root)
    print("train max_x:", max_x)
    print("train max_y:", max_y)

    train_ds = JointFrameDataset(train_df, cfg.dataset_root, max_x, max_y)
    val_ds = JointFrameDataset(val_df, cfg.dataset_root, max_x, max_y)
    test_ds = JointFrameDataset(test_df, cfg.dataset_root, max_x, max_y)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

    model = JointMLP(
        input_dim=28,
        hidden_dim1=cfg.hidden_dim1,
        hidden_dim2=cfg.hidden_dim2,
        dropout=cfg.dropout,
        num_classes=3
    ).to(cfg.device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val_f1 = -1
    ckpt_path = os.path.join(cfg.save_dir, f"best_joint_mlp_xyonly_{cfg.joint_source}.pth")

    for epoch in range(cfg.num_epochs):
        tr_loss, tr_acc, tr_f1 = train_one_epoch(model, train_loader, optimizer, criterion, cfg.device)
        va_loss, va_acc, va_f1, va_cm= evaluate(model, val_loader, criterion, cfg.device)

        print(f"\nEpoch {epoch+1}/{cfg.num_epochs}")
        print_metrics("Train", tr_loss, tr_acc, tr_f1)
        print_metrics("Val", va_loss, va_acc, va_f1)

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "max_x": max_x,
                    "max_y": max_y,
                    "config": cfg.__dict__
                },
                ckpt_path
            )
            print("saved:", ckpt_path)

    print("\nLoading best model for test...")
    ckpt = torch.load(ckpt_path, map_location=cfg.device)
    model.load_state_dict(ckpt["model_state_dict"])

    te_loss, te_acc, te_f1, te_cm = evaluate(model, test_loader, criterion, cfg.device)
    print_metrics("Test", te_loss, te_acc, te_f1)
    print("\nConfusion matrix:")
    print(te_cm)


if __name__ == "__main__":
    main()

train samples: 3195
val samples  : 675
test samples : 720
train max_x: 519.7807991120978
train max_y: 893.7252591894439

Epoch 1/20

[Train]
Loss     : 0.9504
Accuracy : 0.6782
Macro F1 : 0.6809

[Val]
Loss     : 0.6380
Accuracy : 0.9719
Macro F1 : 0.9718
saved: ./checkpoints_joint_mlp_v2\best_joint_mlp_xyonly_RGB.pth

Epoch 2/20

[Train]
Loss     : 0.4066
Accuracy : 0.9017
Macro F1 : 0.9018

[Val]
Loss     : 0.1758
Accuracy : 0.9674
Macro F1 : 0.9676

Epoch 3/20

[Train]
Loss     : 0.1896
Accuracy : 0.9515
Macro F1 : 0.9517

[Val]
Loss     : 0.0836
Accuracy : 0.9852
Macro F1 : 0.9852
saved: ./checkpoints_joint_mlp_v2\best_joint_mlp_xyonly_RGB.pth

Epoch 4/20

[Train]
Loss     : 0.1328
Accuracy : 0.9618
Macro F1 : 0.9619

[Val]
Loss     : 0.0538
Accuracy : 0.9881
Macro F1 : 0.9881
saved: ./checkpoints_joint_mlp_v2\best_joint_mlp_xyonly_RGB.pth

Epoch 5/20

[Train]
Loss     : 0.1059
Accuracy : 0.9703
Macro F1 : 0.9703

[Val]
Loss     : 0.0452
Accuracy : 0.9867
Macro F1 : 0.9866

Epoch 6